# Did the AI credit the right person? Scoring "who said what"

## Why this notebook exists

Imagine an AI tool that listens to a company's quarterly earnings call and writes a summary:

> *"Tomas Okafor asked whether the new plant in Poland will change capital spending."*

Tomas Okafor is the company's finance chief. He did not ask that. **An analyst, Sofia Marchetti, asked it**, and
he answered it. Nothing in the words is wrong; the *speaker* is wrong. In a summary, a mistake like that puts
a statement in the wrong person's mouth.

Behind such summaries there are two steps: the tool **transcribes** the recording and marks which parts come from the
same voice, then it **works out a name** for each voice, by reading the conversation for clues such as "thanks, Marta" or "over to our CFO". Either step can fail. This repository contains a **scorer**:
a small program that checks a tool's work against a human-checked transcript and says, with numbers, *how often
words were credited to the right person* and *what kinds of mistakes were made*. It can also fail a build
automatically when the score is too low (a "gate").

This notebook shows how the scorer works, one step at a time, on a **made-up** earnings call. You need no accounts,
no keys and no internet. Nothing is hidden: every number can be checked by counting.

**Who it is for:** anyone who wants to see what the scorer does, and anyone who wants to use it on their own
recordings. No programming background is needed to follow the text. The code cells are short and print plain results.

**How to read it:** read the text, look at the result under each code cell, and try the **Predict** boxes before
you run the cell.

**To run it:** see "Running the notebooks" in `AI.md`, or from the repository folder:
`python3 -m venv .venv`, then `.venv/bin/pip install -e ".[notebooks]"`, then
`.venv/bin/jupyter lab notebooks/01-score-a-call.ipynb`. The scorer itself uses only standard Python.

## The words used in this notebook

| Word | Plain meaning |
|---|---|
| **Right answer** | A transcript a human checked: for every turn, *who really spoke* and what they said. |
| **Tool output** | What the AI transcription tool produced for the same recording: the text, plus its answer to who spoke. (In the code and files this is called the `output`.) |
| **Turn** | One stretch of speech by one speaker, like one paragraph in a script. |
| **Voice** | An anonymous label such as `V1`. The tool groups sound-alike speech and gives each group a label. It does not know names yet. |
| **Name** | The name the tool's second step worked out for a voice, such as "Marta Lindqvist". It may be wrong, partial, or missing. |
| **Alias** | Another spelling we accept for a person, such as "Tom Okafor" for "Tomas Okafor". |
| **Placeholder** | A non-name such as "Speaker 5". It means "unknown". |
| **Scorer** | The program in this repository that compares the tool output with the right answer. |

## Step 1: how to prepare your own data

*In one sentence:* the scorer reads two small text files in a format called **JSON**: the right answer and the tool
output. Any recording can be scored once you write those two files (a small converter program can do it for you; see Step 10).

First, the loading tools we need. This cell has no output.

In [1]:
import json

from speaker_aware_eval.whos_who import FormatError, load_output, load_truth, parse_output, parse_truth
from speaker_aware_eval.whos_who import ScorerConfig, score

**The right answer file** lists the people, then every turn with its true speaker:

In [2]:
right_answer_example = {
    "conversation_id": "example-call",
    "people": [
        {"name": "Marta Lindqvist", "aliases": ["Marta", "Chief Executive"]},
        {"name": "Sofia Marchetti"},
    ],
    "turns": [
        {"speaker": "Sofia Marchetti", "text": "Will the new plant change your capital spending?"},
        {"speaker": "Marta Lindqvist", "text": "Not materially."},
    ],
}
print(json.dumps(right_answer_example, indent=2))

{
  "conversation_id": "example-call",
  "people": [
    {
      "name": "Marta Lindqvist",
      "aliases": [
        "Marta",
        "Chief Executive"
      ]
    },
    {
      "name": "Sofia Marchetti"
    }
  ],
  "turns": [
    {
      "speaker": "Sofia Marchetti",
      "text": "Will the new plant change your capital spending?"
    },
    {
      "speaker": "Marta Lindqvist",
      "text": "Not materially."
    }
  ]
}


- `conversation_id`: a label for the call. Both files must use the same one.
- `people` (optional): who is on the call, and the spellings we accept (`aliases`) for each.
- `turns`: in order, the true `speaker` and the `text` they said.

**The tool output file** has no `people`. Each turn says which anonymous **voice** the tool heard, and the **name** the tool
gave that voice (or nothing at all):

In [3]:
tool_output_example = {
    "conversation_id": "example-call",
    "turns": [
        {"voice": "V0", "speaker": "Sofia Marchetti", "text": "Will the new plant change your capital spending?"},
        {"voice": "V1", "speaker": "Speaker 2", "text": "Not materially."},
    ],
}
print(json.dumps(tool_output_example, indent=2))

{
  "conversation_id": "example-call",
  "turns": [
    {
      "voice": "V0",
      "speaker": "Sofia Marchetti",
      "text": "Will the new plant change your capital spending?"
    },
    {
      "voice": "V1",
      "speaker": "Speaker 2",
      "text": "Not materially."
    }
  ]
}


- `voice` (required): the tool's anonymous label.
- `speaker`: the name the tool gave. It can be a real name, a placeholder like "Speaker 2", or empty.
- The text may be cut into turns differently from the right answer. That is fine: the scorer compares *words*, not turns.

If a file does not follow the format, the scorer tells you what is wrong in plain words instead of guessing:

In [4]:
broken_output = {
    "conversation_id": "example-call",
    "turns": [{"speaker": "Sofia Marchetti", "text": "Will the new plant change your capital spending?"}],
}
try:
    parse_output(broken_output)
except FormatError as error:
    print("The scorer says:", error)

The scorer says: system output: turns[0] needs a non-empty 'voice'


To score your own call, save your two files and load them with `load_truth("my_right_answer.json")` and
`load_output("my_tool_output.json")`, as the next step does with ours.

## Step 2: our made-up call

*In one sentence:* a fictional company, **Halvorsen Components**, holds its third-quarter earnings call: an operator
introduces speakers, the chief executive and the finance chief present, and four analysts ask questions. This is a
normal shape for an earnings call, which is exactly the kind of recording where "who said it" matters, and where
a transcription tool often struggles (many voices, over a phone line).

Everything is invented: the company, the people and the words. The files are in `tests/fixtures/whos_who/`.

> **Honest note:** real tools make mistakes at random. For teaching, this call's tool output was written by hand
> to contain **every kind of mistake** the scorer knows, so we can see each one. It is far worse than a good tool would be.

In [5]:
DATA = "../tests/fixtures/whos_who/"

right_answer = load_truth(DATA + "earnings_call_truth.json")
tool_output = load_output(DATA + "earnings_call_output.json")

print("Call:", right_answer.conversation_id)
print("Turns in the right answer:", len(right_answer.turns))
print("Turns in the tool output :", len(tool_output.turns))

Call: halvorsen-q3
Turns in the right answer: 23
Turns in the tool output : 23


**The right answer**, as a script. `Speaker 1` at turn 18 is a voice the human transcriber could not identify
(someone asking to repeat a number): a **placeholder** in the right answer.

In [6]:
number = 1
for turn in right_answer.turns:
    print(number, "|", turn.speaker, "|", turn.text)
    number = number + 1

1 | Operator | Good morning, and welcome to the Halvorsen Components third quarter earnings call. All participants are in listen-only mode. I will now hand the call to Marta Lindqvist, our chief executive.
2 | Marta Lindqvist | Thank you, operator, and good morning everyone. Revenue grew eight percent this quarter, driven by strong demand for our industrial sensors.
3 | Marta Lindqvist | We also opened our new plant in Poland, which is on budget and ahead of schedule.
4 | Tomas Okafor | Thanks, Marta. Gross margin was thirty eight percent, up two points, and we ended the quarter with strong cash flow.
5 | Marta Lindqvist | Looking ahead, we now expect full year revenue growth of seven to nine percent, and we are raising our outlook for the sensor business.
6 | Operator | Our first question comes from Priya Raman at Fenwick Securities.
7 | Priya Raman | Thanks for taking my question. Can you say more about demand from automotive customers?
8 | Marta Lindqvist | Automotive orders rose tw

**The tool output**, for the same call. Compare it with the script above. Each line shows the voice the tool heard,
the name it settled on, and the text. Some names are right and some are not; try to spot them before reading on.

In [7]:
number = 1
for turn in tool_output.turns:
    print(number, "|", turn.voice, "|", turn.speaker, "|", turn.text)
    number = number + 1

1 | V0 | Operator | Good morning, and welcome to the Halvorsen Components third quarter earnings call. All participants are in listen-only mode. I will now hand the call to Marta Lindqvist, our chief executive.
2 | V1 | Lindqvist | Thank you, operator, and good morning everyone. Revenue grew eight percent this quarter, driven by strong demand for our industrial sensors.
3 | V1 | Lindqvist | We also opened our new plant in Poland, which is on budget and ahead of schedule.
4 | V2 | Tom Okafor | Thanks, Marta. Gross margin was thirty eight percent, up two points, and we ended the quarter with strong cash flow.
5 | V5 | Speaker 5 | Looking ahead, we now expect full year revenue growth of seven to nine percent, and we are raising our outlook for the sensor business.
6 | V0 | Operator | Our first question comes from Priya Raman at Fenwick Securities.
7 | V3 | Daniel Whitcombe | Thanks for taking my question. Can you say more about demand from automotive customers?
8 | V1 | Lindqvist | Automo

## Step 3: the big picture, in one run

*In one sentence:* the scorer takes the two files and returns a report; the most important number is the **headline
score**: out of all the words it could compare, the share that were credited to the right person.

```
 right answer (who really spoke)        tool output (voices + names it chose)
                \                          /
                 1. match the words up
                 2. work out which person each voice really is
                 3. label every matched word: correct, or one kind of mistake
                 4. headline score, coverage, pass/fail gate
```

Let us run it once and then open each step.

In [8]:
report = score(right_answer, tool_output)

compared = report["coverage"]["aligned_words"]
correct = report["words"]["by_label"]["correct"]

print("Words compared          :", compared)
print("Credited to right person:", correct)
print("Headline score          :", report["headline_score"], "  (that is", correct, "divided by", compared, ")")
print("Needed to pass the gate :", report["gate"]["min_score"])
print("Gate result             :", "PASS" if report["gate"]["passed"] else "FAIL")

Words compared          : 308
Credited to right person: 209
Headline score          : 0.678571   (that is 209 divided by 308 )
Needed to pass the gate : 0.9
Gate result             : FAIL


The tool failed the gate: only about two thirds of the words were credited to the right person. The next
steps explain where the other third went.

## Step 4: names, aliases and placeholders

*In one sentence:* every name the tool gives is sorted into one of five kinds: **full** (the right person, or an
accepted alias), **partial** (part of one person's name, like "Lindqvist"), **ambiguous** (part of a name that fits two
or more people on the call, so it points to no one), **placeholder** (no real name, like "Speaker 5"),
or **unknown** (a name that is nobody on this call).

Names are compared by their words. Capital letters, punctuation and word order do not matter. The scorer
does **not** guess that a near-miss spelling is the same person: only the accepted aliases count.

`roster_for` gives the scorer's own list of the people on this call and the spellings it accepts, so the answers
below are exactly how the scorer reads each name.

In [9]:
from speaker_aware_eval.whos_who import roster_for

roster = roster_for(right_answer)

names_the_tool_might_give = [
    "Marta Lindqvist",
    "Tom Okafor",
    "okafor TOMAS",
    "Lindqvist",
    "Speaker 5",
    "Hugo Brant",
]
for name in names_the_tool_might_give:
    answer = roster.resolve(name)
    who = answer.person_name
    if who is None:
        who = "nobody"
    print(name, "->", answer.kind, "->", who)

Marta Lindqvist -> full -> Marta Lindqvist
Tom Okafor -> full -> Tomas Okafor
okafor TOMAS -> full -> Tomas Okafor
Lindqvist -> partial -> Marta Lindqvist
Speaker 5 -> placeholder -> nobody
Hugo Brant -> unknown -> nobody


- "Tom Okafor" is an accepted alias, so it counts as the full name.
- "okafor TOMAS" has the same words as "Tomas Okafor", so it matches too.
- "Lindqvist" is only part of Marta Lindqvist's name: **partial**. By default a partial name still counts as correct (Step 8 shows what changes if it does not).
- "Speaker 5" is a **placeholder**.
- "Hugo Brant" is spelled differently from "Hugo Brandt" and is not an accepted alias, so it is **unknown**. The scorer will treat it as a name that is not on the call. This is deliberate (see "Limits").
- This call has no two people who share a first name. If it had, a bare first name would be **ambiguous**: the scorer does not
  guess which person was meant. Like an unknown name, it is counted as `wrong_name`.

## Step 5: matching the words up

*In one sentence:* the two texts are compared word by word, like the "compare documents" feature of a word processor.
Only words that match are scored. Voices and names play no part yet.

Why: the tool may mishear or cut turns differently, and "who said it" only makes sense for words both sides
agree were spoken.

To see it, we copy the tool output and change turn 7 as if the tool had misheard it. Then we ask the scorer for
its own word-by-word matching with `align`. Before comparing, the scorer drops capital letters and punctuation.

In [10]:
from speaker_aware_eval.whos_who import align

misheard_output = json.load(open(DATA + "earnings_call_output.json"))
misheard_output["turns"][6]["text"] = "Thanks for taking my questions. Can you say about demand from automotive customers?"
misheard = parse_output(misheard_output)

print("Right answer, turn 7:", right_answer.turns[6].text)
print("Tool output,  turn 7:", misheard.turns[6].text)
print()
for word in align(right_answer, misheard):
    if word.truth_turn == 7 or word.output_turn == 7:
        if word.truth_word == word.output_word:
            print("same              :", word.truth_word)
        elif word.output_word is None:
            print("only right answer :", word.truth_word)
        else:
            print("only tool output  :", word.output_word)

Right answer, turn 7: Thanks for taking my question. Can you say more about demand from automotive customers?
Tool output,  turn 7: Thanks for taking my questions. Can you say about demand from automotive customers?

same              : thanks
same              : for
same              : taking
same              : my
only right answer : question
only tool output  : questions
same              : can
same              : you
same              : say
only right answer : more
same              : about
same              : demand
same              : from
same              : automotive
same              : customers


Words that are the same are scored. The others are **not** scored: they lower **coverage** (how much of the
right answer could be matched), never the who's-who score itself. This is the matching the scorer itself used:
`align` returns it straight from the scorer, with the label of every scored word (Step 7 uses that).

*Predict:* with these words unmatched, does coverage stay at 100%? Run the next cell.

In [11]:
misheard_report = score(right_answer, misheard)

print("Coverage, original call    :", report["coverage"]["share"])
print("Coverage, one turn misheard:", misheard_report["coverage"]["share"])

Coverage, original call    : 1.0
Coverage, one turn misheard: 0.993506


Coverage drops slightly, because some words no longer match. The score is calculated only over the words that still match.
Always read coverage next to the score: a tool that drops half the call could otherwise look good.

## Step 6: which person is each voice?

*In one sentence:* for each voice, the scorer counts whose words it actually carries (using the right answer) and
the person with most of them is the voice's real person. Separately it notes which name the tool gave.

Mixed-up voices show here: a voice with two real people in it, one person spread over two voices, or two voices
carrying each other's names. (`None` below means the tool gave no usable real name: a placeholder, nothing, or
a name that is nobody on the call.)

In [12]:
for voice, info in report["voice_map"].items():
    print("Voice", voice, "is really", info["person"], "| the tool named it:", info["named_as"], "| mixed:", info["merged"])
    for person, share in info["shares"].items():
        print("      ", round(share * 100), "% of this voice's words are", person)

Voice V0 is really Operator | the tool named it: Operator | mixed: False
       96 % of this voice's words are Operator
       4 % of this voice's words are Tomas Okafor
Voice V1 is really Marta Lindqvist | the tool named it: Marta Lindqvist | mixed: False
       100 % of this voice's words are Marta Lindqvist
Voice V2 is really Tomas Okafor | the tool named it: Tomas Okafor | mixed: True
       77 % of this voice's words are Tomas Okafor
       23 % of this voice's words are Sofia Marchetti
Voice V5 is really Marta Lindqvist | the tool named it: None | mixed: False
       100 % of this voice's words are Marta Lindqvist
Voice V3 is really Priya Raman | the tool named it: Daniel Whitcombe | mixed: False
       100 % of this voice's words are Priya Raman
Voice V4 is really Daniel Whitcombe | the tool named it: Priya Raman | mixed: False
       100 % of this voice's words are Daniel Whitcombe
Voice V6 is really Hugo Brandt | the tool named it: None | mixed: False
       100 % of this voic

How to read it:

- **V2** carries the finance chief (Tomas Okafor) *and* an analyst (Sofia Marchetti), who spoke only briefly: the tool treated two people as one voice. The scorer marks it **mixed** because both people are a large share of the voice.
- **V3** is really Priya Raman but the tool named it "Daniel Whitcombe"; **V4** is really Daniel but was named "Priya Raman". The names are **swapped**.
- **V1** and **V5** are both the chief executive: one person split across two voices. The second voice was named "Speaker 5", a placeholder.
- **V6** is really Hugo Brandt but the tool wrote "Hugo Brant" (a misspelling, so nobody on the call): named `None`. **V8** is Anika Sen, named "Speaker 6": also `None`.
- **V0** is the operator and is named correctly, but it also carries a few words of the finance chief's goodbye (turn 23): too few to count as mixed.

## Step 7: every kind of mistake

*In one sentence:* every matched word that is not correct gets **one** label. If more than one could apply, the
first in this list wins: `merged`, `split`, `swap`, `partial`, `unnamed`, `wrong_name`, `misattributed_turn`.

| Label | Plain meaning | Where it happens in our call |
|---|---|---|
| `merged` | One voice mixes two people (each is at least 20% of it); this word belongs to the voice's *minor* person. | Turn 13: Sofia's question, on the finance chief's voice |
| `split` | One person speaks on two voices (each at least 20% of their words); this word is on the *less used* voice. | Turn 5: the chief executive on voice V5 |
| `swap` | Two voices carry each other's names. | Turns 7 and 10: Priya and Daniel |
| `partial` | Only part of the right name was given. | Not by default: Step 8 |
| `unnamed` | The tool gave no real name (placeholder or empty). | Turn 20: Anika Sen named "Speaker 6" |
| `wrong_name` | The name is nobody on the call, or it is part of two or more people's names, so it points to no one. | Turn 16: "Hugo Brant" |
| `misattributed_turn` | A different real person's name was given, with no merge, split or swap behind it. | Turn 23: the finance chief's goodbye, named as the operator |

*Predict:* turn 5 was named "Speaker 5", a placeholder, so `unnamed` could apply. But `split` is earlier in the list.
Which label does turn 5 get? Run the next cell.

In [13]:
for turn in report["turns"]["detail"]:
    line = str(turn["turn"]) + " | " + turn["speaker"] + " | " + turn["label"]
    if turn["also"]:
        line = line + "   (also applies: " + ", ".join(turn["also"]) + ")"
    print(line)

1 | Operator | correct
2 | Marta Lindqvist | correct
3 | Marta Lindqvist | correct
4 | Tomas Okafor | correct
5 | Marta Lindqvist | split   (also applies: unnamed)
6 | Operator | correct
7 | Priya Raman | swap   (also applies: misattributed_turn)
8 | Marta Lindqvist | correct
9 | Operator | correct
10 | Daniel Whitcombe | swap   (also applies: misattributed_turn)
11 | Tomas Okafor | correct
12 | Operator | correct
13 | Sofia Marchetti | merged
14 | Tomas Okafor | correct
15 | Operator | correct
16 | Hugo Brandt | wrong_name
17 | Tomas Okafor | correct
19 | Operator | correct
20 | Anika Sen | unnamed
21 | Tomas Okafor | correct
22 | Marta Lindqvist | correct
23 | Tomas Okafor | misattributed_turn


Turn 5 is `split`, and the line tells you `unnamed` would also have applied. Turns labelled `correct` include a full name,
an alias ("Tom Okafor", turn 4) and a partial name that counts ("Lindqvist"). Turn 18 is missing: it is the placeholder
speaker in the right answer, which is left out of the score.

Behind each turn's label are the labels of its words, and `align` gives them one by one. Here is turn 23, the
finance chief's goodbye:

In [14]:
for word in align(right_answer, tool_output):
    if word.truth_turn == 23 and word.status == "scored":
        print(word.truth_word, "| said by", word.person, "| voice", word.voice, "named", word.name, "|", word.label, "| also:", word.also)

thanks | said by Tomas Okafor | voice V0 named Operator | misattributed_turn | also: ()
everyone | said by Tomas Okafor | voice V0 named Operator | misattributed_turn | also: ()
goodbye | said by Tomas Okafor | voice V0 named Operator | misattributed_turn | also: ()


All three words were said by Tomas Okafor but sit on the operator's voice, which the tool named "Operator":
`misattributed_turn`. No other label applied, so `also` is empty. When the words of one turn get different labels,
the turn's label is the most common one (a tie goes to the label earlier in the list above).

Now the same information per person, so a good average cannot hide a person who was always wrong:

In [15]:
for person in report["per_person"]:
    errors = {}
    for label, count in person["errors"].items():
        if count > 0:
            errors[label] = count
    print(person["person"], "|", person["correct_words"], "of", person["aligned_words"], "words right |", errors)

Operator | 77 of 77 words right | {}
Marta Lindqvist | 57 of 81 words right | {'split': 24}
Tomas Okafor | 75 of 78 words right | {'misattributed_turn': 3}
Priya Raman | 0 of 14 words right | {'swap': 14}
Daniel Whitcombe | 0 of 16 words right | {'swap': 16}
Sofia Marchetti | 0 of 22 words right | {'merged': 22}
Hugo Brandt | 0 of 9 words right | {'wrong_name': 9}
Anika Sen | 0 of 11 words right | {'unnamed': 11}


## Step 8: the `partial` label

*In one sentence:* by default a partial name ("Lindqvist") counts as correct. If you decide it should not, the same
words become `partial` errors.

In [16]:
strict_report = score(right_answer, tool_output, ScorerConfig(partial_is_correct=False))

print("Headline, partial names count as correct:", report["headline_score"])
print("Headline, partial names are errors      :", strict_report["headline_score"])
print()
print("Words per label with strict partial names:")
for label, count in strict_report["words"]["by_label"].items():
    print("  ", label, count)

Headline, partial names count as correct: 0.678571
Headline, partial names are errors      : 0.493506

Words per label with strict partial names:
   correct 152
   merged 22
   split 24
   swap 30
   partial 57
   unnamed 11
   wrong_name 9
   misattributed_turn 3


Now **all seven kinds of mistake** appear in one call.

## Step 9: the report and the pass/fail gate on the command line

*In one sentence:* from a terminal you can score two files with one command; it prints a plain-text report and ends
with a status code a build system can act on: **0** gate passed, **1** gate failed, **2** the input was bad.

```
python -m speaker_aware_eval.whos_who  right_answer.json  tool_output.json
```

Here we run that exact command from inside the notebook, on our files. `sys.executable` stands for the Python
that runs this notebook.

In [17]:
import subprocess
import sys

command = [
    sys.executable, "-m", "speaker_aware_eval.whos_who",
    DATA + "earnings_call_truth.json", DATA + "earnings_call_output.json",
]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
print("Status code:", result.returncode, "(1 means the gate failed)")

Who's-who report: halvorsen-q3
Headline score: 67.9% (coverage 100.0%): 209 of 308 aligned words credited to the right person
Gate: FAIL - score 67.9% (coverage 100.0%), bar 90.0%
Coverage: 100.0% of right-answer words aligned (0 right-answer and 0 system words not aligned, not scored)
Placeholder voices left out of the score: 7 words in 1 turns (7 system words lined up with them and are not counted as unaligned)
Errors (words / turns):
  named as another person: 3 / 1
  names swapped: 30 / 2
  name not matched to one person in the call: 9 / 1
  no name (placeholder): 11 / 1
  mixed into another person's voice: 22 / 1
  second voice for the same person: 24 / 1
Partial names counted as correct: 57 words
Per person:
  Operator: 100.0% (77 of 77 words; 25.0% of speech)
  Marta Lindqvist: 70.4% (57 of 81 words; 26.3% of speech)
  Tomas Okafor: 96.2% (75 of 78 words; 25.3% of speech)
  Priya Raman: 0.0% (0 of 14 words; 4.5% of speech)
  Daniel Whitcombe: 0.0% (0 of 16 words; 5.2% of speech)

Options exist for the pass mark (`--min-score`), a coverage bar (`--min-coverage`, off by default; it makes the gate fail when too
little of the right answer was matched, however good the score), strict partial names, and so on; `--help` lists them. With a lower pass mark
the same call passes:

In [18]:
result = subprocess.run(command + ["--min-score", "0.6"], capture_output=True, text=True)
print("Status code:", result.returncode, "(0 means the gate passed)")

Status code: 0 (0 means the gate passed)


## Step 10: where real data comes from

*In one sentence:* real tools produce their own formats. Small converters ("adapters") turn them into the two files
from Step 1. All examples below are made up.

**A right answer from a published transcript.** One format, used by a public earnings-call dataset, is a table with one
word per row and a separate list of speaker names. The adapter turns it into turns. If the list has no name for a speaker, the adapter stops with an error that names that speaker,
instead of guessing; pass `unnamed_speakers="placeholder"` to keep the speaker as a placeholder voice, which the scorer leaves out of
the score:

In [19]:
from speaker_aware_eval.whos_who.adapters import build_output, build_truth, deepgram, scribe

truth_from_table = build_truth(DATA + "earnings21/9000001.nlp", DATA + "earnings21/speaker-metadata.csv")
for turn in truth_from_table["turns"]:
    print(turn["speaker"], "|", turn["text"])

Alex Morgan | Good morning everyone, welcome to the call.
Dana Reyes | This is Dana Reyes.


**Tool output from a transcript with voice labels, plus a naming step.** A transcription service returns text with voice labels
(`SPEAKER_00`, ...). A separate naming step then says which name goes with which label; `null` means it could not tell.

In [20]:
voices_and_text = json.load(open(DATA + "diarized/sample_chunks.json"))
naming = json.load(open(DATA + "diarized/sample_naming.json"))
print("Naming step result:", naming)

output_from_service = build_output(voices_and_text, naming, conversation_id="9000001")
for turn in output_from_service["turns"]:
    print(turn["voice"], "|", turn["speaker"], "|", turn["text"])

small_report = score(
    parse_truth(truth_from_table),
    parse_output(output_from_service),
)
print()
print("Headline:", small_report["headline_score"], "| coverage:", small_report["coverage"]["share"])

Naming step result: {'SPEAKER_00': 'Alex Morgan', 'SPEAKER_01': 'Dana Reyes', 'SPEAKER_02': None}
SPEAKER_00 | Alex Morgan | Good morning everyone, welcome to the call.
SPEAKER_01 | Dana Reyes | This is Dana Reyes.
SPEAKER_02 | None | Thanks.

Headline: 1.0 | coverage: 1.0


The last turn ("Thanks.") is not in the right answer, so it is not matched and not scored.

Two commercial services' own formats (Deepgram and ElevenLabs Scribe) are handled the same way: convert, then name.
Tiny made-up examples, cut down to the fields the converters read:

In [21]:
deepgram_result = {"results": {"channels": [{"alternatives": [{"words": [
    {"word": "hello", "punctuated_word": "Hello,", "speaker": 0},
    {"word": "everyone", "punctuated_word": "everyone.", "speaker": 0},
    {"word": "hi", "punctuated_word": "Hi.", "speaker": 1},
]}]}]}}

scribe_result = {"words": [
    {"text": "Hello,", "type": "word", "speaker_id": "speaker_0"},
    {"text": " ", "type": "spacing", "speaker_id": "speaker_0"},
    {"text": "everyone.", "type": "word", "speaker_id": "speaker_0"},
    {"text": "Hi.", "type": "word", "speaker_id": "speaker_1"},
]}

print("Deepgram:", deepgram.to_diarized(deepgram_result))
print("Scribe  :", scribe.to_diarized(scribe_result))

Deepgram: {'speaker_turns': [{'speaker': 'SPEAKER_0', 'text': 'Hello, everyone.'}, {'speaker': 'SPEAKER_1', 'text': 'Hi.'}]}
Scribe  : {'speaker_turns': [{'speaker': 'SPEAKER_0', 'text': 'Hello, everyone.'}, {'speaker': 'SPEAKER_1', 'text': 'Hi.'}]}


## Step 11: check it yourself

*In one sentence:* every number above can be rebuilt by counting. These checks stop the notebook with an error if any
claim made in the text stops being true.

In [22]:
from speaker_aware_eval.whos_who import ERROR_LABELS

# 1. All seven kinds of mistake appeared (partial needs the strict setting).
for label in ERROR_LABELS:
    assert strict_report["words"]["by_label"][label] > 0, label

# 2. Every matched word has exactly one label, and the headline is correct words divided by matched words.
total_words = 0
for count in report["words"]["by_label"].values():
    total_words = total_words + count
assert total_words == compared
assert report["headline_score"] == round(correct / compared, 6)

# 3. Rebuild the operator's score by adding up that person's turns by hand.
operator_correct = 0
for turn in report["turns"]["detail"]:
    if turn["speaker"] == "Operator":
        operator_correct = operator_correct + turn["aligned_words"] - turn["wrong_words"]
operator = [p for p in report["per_person"] if p["person"] == "Operator"][0]
assert operator["correct_words"] == operator_correct

# 4. The predictions made in the text.
expected = {
    5: "split", 7: "swap", 10: "swap", 13: "merged", 16: "wrong_name",
    20: "unnamed", 23: "misattributed_turn",
}
actual = {}
for turn in report["turns"]["detail"]:
    if turn["label"] not in ("correct", "partial"):
        actual[turn["turn"]] = turn["label"]
assert actual == expected, actual

# 5. The scorer's own word matching adds up to the report: count the label of every scored word.
scored_labels = {}
for word in align(right_answer, tool_output):
    if word.status == "scored":
        scored_labels[word.label] = scored_labels.get(word.label, 0) + 1
for label, count in report["words"]["by_label"].items():
    assert scored_labels.get(label, 0) == count, label

print("All checks passed")

All checks passed


## Limits, and where to go next

Read these before quoting any number from a scorer like this.

- **Spelling is reviewed by people.** The scorer never guesses that "Brant" means "Brandt". Add accepted spellings as
  aliases. The first cell below shows what that does for Hugo Brandt.
- **A messy reference name can fail to match.** Real transcripts sometimes carry a stray "'s" after a name. The second cell shows
  the effect with made-up names. The scorer does not clean names; it credits aliases someone reviewed.
- **Only matched words are scored.** Read coverage (Step 5) next to the score. If you want the gate to demand coverage too, set
  `--min-coverage`.
- **The 20% limit for mixed voices and the 90% pass mark are choices**, not facts. They are settings, and the report records them.
- **A voice that holds two people can still score 100%.** The score credits each word to the name the tool gave, turn by turn.
  If every turn is named correctly, the score is 100% even when one voice carries two people. The report still marks that voice as
  `merged` (Step 6), so read it next to the score.
- **Matching gets slow on repetitive text.** Time grows fast when a transcript repeats a few words over and over. On one laptop,
  10,000 words took under half a second with 2,000 different words and about 8 seconds with 20. Normal speech is in the fast range;
  a broken output that loops on a phrase is not. The scorer does not stop you, so check what you feed it.
- **One call shows that the machinery works, not that a tool is good.** Measuring a tool needs many real calls.

In [23]:
# Accept "Hugo Brant" as a spelling of Hugo Brandt, then score again.
truth_with_alias = json.load(open(DATA + "earnings_call_truth.json"))
for person in truth_with_alias["people"]:
    if person["name"] == "Hugo Brandt":
        person["aliases"] = ["Hugo Brant"]

fixed_report = score(parse_truth(truth_with_alias), tool_output)

for person in report["per_person"]:
    if person["person"] == "Hugo Brandt":
        print("Hugo Brandt's words wrong before:", person["errors"]["wrong_name"])
for person in fixed_report["per_person"]:
    if person["person"] == "Hugo Brandt":
        print("Hugo Brandt's words wrong after :", person["errors"]["wrong_name"])
print("Headline before:", report["headline_score"], "| after:", fixed_report["headline_score"])

Hugo Brandt's words wrong before: 9
Hugo Brandt's words wrong after : 0
Headline before: 0.678571 | after: 0.707792


In [24]:
# A reference name with a stray "'s" after it, with and without an accepted alias.
def score_small_call(reference_name, accepted_alias=None):
    people = [{"name": reference_name}, {"name": "Omar Haddad"}]
    if accepted_alias:
        people[0]["aliases"] = [accepted_alias]
    right = parse_truth({
        "conversation_id": "x", "people": people,
        "turns": [{"speaker": reference_name, "text": "hello there today"},
                  {"speaker": "Omar Haddad", "text": "okay then"}],
    })
    tool = parse_output({
        "conversation_id": "x",
        "turns": [{"voice": "A", "speaker": "Dana Reyes", "text": "hello there today"},
                  {"voice": "B", "speaker": "Omar Haddad", "text": "okay then"}],
    })
    return score(right, tool, ScorerConfig(partial_is_correct=False))["headline_score"]

print("Reference name \"Dana Reyes's\", no alias  :", score_small_call("Dana Reyes's"))
print("Reference name \"Dana Reyes's\", with alias:", score_small_call("Dana Reyes's", "Dana Reyes"))

Reference name "Dana Reyes's", no alias  : 0.4
Reference name "Dana Reyes's", with alias: 1.0


**To explore on your own:** open `tests/fixtures/whos_who/earnings_call_output.json`, change a name or move a turn to
a different voice, and predict the new labels before you re-run Steps 3 to 7. Or score your own two files with the command
in Step 9.